# 💬 NLTK — NLP
## Python Ecosystem Tutorial Series — Module 10 of 18

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

| | |
|---|---|
| **Library** | 💬 NLTK |
| **Domain** | NLP |
| **Dataset** | Drug safety abstracts |
| **Module** | 10 of 18 |

**What you will learn:**

1. What NLTK is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install nltk
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `word_tokenize()` | Split into words |
| `sent_tokenize()` | Split into sentences |
| `stopwords.words()` | Common words to remove |
| `PorterStemmer()` | Reduce word endings |
| `pos_tag()` | Label parts of speech |

# 10. 💬 NLTK — Natural Language Processing
> **Python + NLTK = NLP**

NLTK (Natural Language Toolkit) provides tools for processing human language:
tokenisation, stemming, POS tagging, sentiment analysis, named entity recognition.

**Key concepts:** tokenise, stopwords, stemming/lemmatisation, POS tags, frequency distribution

In [ ]:
import nltk
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Download required NLTK data (only once)
for resource in ["punkt","stopwords","averaged_perceptron_tagger",
                  "wordnet","vader_lexicon","punkt_tab"]:
    try:
        nltk.download(resource, quiet=True)
    except:
        pass

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
try:
    from nltk.sentiment import SentimentIntensityAnalyzer
    SIA_OK = True
except:
    SIA_OK = False

# ── Real text: drug safety abstract ──────────────────────────────────────────
abstracts = [
    """Acetaminophen is widely used as an analgesic and antipyretic agent.
    Hepatotoxicity represents the most serious adverse effect, particularly following
    overdose. The liver injury involves reactive metabolite NAPQI formation via CYP2E1.
    N-acetylcysteine provides effective antidotal treatment when administered early.
    At therapeutic doses, acetaminophen is generally safe and well-tolerated.""",

    """PFOA is a persistent organic pollutant detected in human blood worldwide.
    The compound shows strong bioaccumulation in liver, kidney and thyroid tissues.
    Epidemiological studies demonstrate associations with cancer, thyroid disease and
    immune dysfunction. The EPA has classified PFOA as a likely human carcinogen.
    Regulatory restrictions have been implemented globally to phase out PFOA production.""",

    """Aspirin inhibits COX-1 and COX-2 enzymes, reducing prostaglandin synthesis.
    Low-dose aspirin therapy effectively prevents cardiovascular events and thrombosis.
    Gastrointestinal bleeding represents the primary adverse effect at higher doses.
    The drug demonstrates excellent efficacy and safety at doses of 75-325 mg daily.
    Long-term aspirin therapy requires careful benefit-risk assessment in patients.""",
]

print("── Text analysis on 3 drug abstracts ──")
print(f"Abstract 1 (first 100 chars): {abstracts[0][:100].strip()}")

In [ ]:
# ── Tokenisation ─────────────────────────────────────────────────────────────
all_text = " ".join(abstracts)
print("── Tokenisation ──")
try:
    tokens = word_tokenize(all_text.lower())
except:
    tokens = re.findall(r"\b[a-z]+\b", all_text.lower())
print(f"Total tokens: {len(tokens)}")
print(f"Sample: {tokens[:15]}")

# ── Remove stopwords ─────────────────────────────────────────────────────────
try:
    stop_words = set(stopwords.words("english"))
except:
    stop_words = {"the","a","an","is","are","was","were","be","been","has",
                   "have","had","do","does","did","and","or","but","in","on",
                   "at","to","for","of","with","as","by","from","that","this","it"}

clean_tokens = [t for t in tokens if t.isalpha() and t not in stop_words and len(t) > 2]
print(f"\n── After removing stopwords: {len(clean_tokens)} tokens")

# ── Word frequency ────────────────────────────────────────────────────────────
freq = Counter(clean_tokens)
print("\n── Top 15 words ──")
for word, count in freq.most_common(15):
    print(f"  {word:20s}: {count}")

# ── Stemming vs Lemmatisation ─────────────────────────────────────────────────
ps  = PorterStemmer()
try:
    lemm = WordNetLemmatizer()
    lemm_ok = True
except:
    lemm_ok = False

test_words = ["running","studies","toxicity","hepatotoxic","effectively","doses"]
print("\n── Stemming vs Lemmatisation ──")
print(f"  {"Word":20s} {"Stemmed":20s} {"Lemmatised":20s}")
for w in test_words:
    stem = ps.stem(w)
    lem  = lemm.lemmatize(w) if lemm_ok else w
    print(f"  {w:20s} {stem:20s} {lem:20s}")

# ── Sentiment Analysis ────────────────────────────────────────────────────────
if SIA_OK:
    sia = SentimentIntensityAnalyzer()
    print("\n── Sentiment scores ──")
    titles = ["Acetaminophen", "PFOA", "Aspirin"]
    sentiments = []
    for title, text in zip(titles, abstracts):
        score = sia.polarity_scores(text)
        sentiments.append(score)
        print(f"  {title}: compound={score['compound']:.3f} (pos={score['pos']:.2f}, neg={score['neg']:.2f})")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word frequency bar
top15 = freq.most_common(15)
words, counts = zip(*top15)
cols_wd = plt.cm.viridis(np.linspace(0.2, 0.9, 15))
axes[0].barh(list(words)[::-1], list(counts)[::-1], color=cols_wd, alpha=0.85, edgecolor="white")
axes[0].set_title("Top 15 Words in Drug Safety Abstracts", fontweight="bold")
axes[0].set_xlabel("Frequency")
axes[0].grid(True, alpha=0.3, axis="x")

# Sentiment bars
if SIA_OK:
    x   = np.arange(3)
    w_s = 0.25
    axes[1].bar(x-w_s, [s["pos"] for s in sentiments],   w_s, label="Positive", color="#27AE60", alpha=0.85)
    axes[1].bar(x,     [s["neg"] for s in sentiments],    w_s, label="Negative", color="#E74C3C", alpha=0.85)
    axes[1].bar(x+w_s, [s["compound"] for s in sentiments],w_s,label="Compound", color="#3498DB", alpha=0.85)
    axes[1].set_xticks(x); axes[1].set_xticklabels(["Acetaminophen","PFOA","Aspirin"])
    axes[1].set_title("Sentiment Analysis of Drug Abstracts", fontweight="bold")
    axes[1].set_ylabel("Score"); axes[1].legend(); axes[1].grid(True, alpha=0.3, axis="y")
else:
    axes[1].bar(["Acetaminophen","PFOA","Aspirin"],[len(a.split()) for a in abstracts],
                color=["#3498DB","#E74C3C","#27AE60"], alpha=0.85, edgecolor="white")
    axes[1].set_title("Word Count per Abstract", fontweight="bold")
    axes[1].set_ylabel("Words")

plt.suptitle("NLTK — Natural Language Processing on Drug Abstracts", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("nltk_nlp.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: NLTK

### The Complete NLP Pipeline
```
Raw text
  -> Sentence tokenise (sent_tokenize)
  -> Word tokenise (word_tokenize)
  -> Remove stopwords ("the", "a", "is", ...)
  -> Stem or Lemmatise (reduce word forms)
  -> POS tag (identify noun/verb/adj)
  -> Analyse (frequency, sentiment, NER)
```

### Stemming vs Lemmatisation
PorterStemmer chops word endings with rules — fast but produces non-words ("studi", "hepatotox"). WordNetLemmatizer looks up a vocabulary to find the base form — slower but always produces valid words ("study", "hepatotoxic").

### VADER Sentiment Scores
Designed for short, informal, and scientific text. Four scores returned:
- pos, neu, neg: proportion of text in each sentiment
- compound: normalised score from -1 (most negative) to +1 (most positive)
- compound > 0.05 = positive, < -0.05 = negative

Drug safety text tends to be neutral-to-negative (reporting adverse effects).

### POS Tags Reference
NN=noun, VBZ=verb present, JJ=adjective, RB=adverb, DT=determiner, IN=preposition.
Filter to only nouns to extract drug/protein names. Filter to verbs to extract biological processes.

### Named Entity Recognition
```python
from nltk import ne_chunk
chunks = ne_chunk(pos_tag(word_tokenize(text)))
# Extracts: ORGANIZATION, PERSON, GPE, FACILITY, DATE
```


## ✅ Key Takeaways — 💬 NLTK

1. Always build an NLP pipeline: tokenise → clean → normalise → analyse
2. Lemmatisation is slower than stemming but produces real words
3. VADER sentiment works well on scientific text without training
4. Filter to nouns for entity extraction, verbs for process extraction

---
*Next: Continue to Module 11 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [himanshugoel.github.io](https://himanshugoel.github.io)*